# DurakZero Vast.ai Training Bootstrap

This notebook prepares a fresh [`vastai/pytorch`](https://hub.docker.com/r/vastai/pytorch/) instance for high-throughput DurakZero self-play training. Update the configuration cell with your fork URL, branch name, GPU usage plan, and the path where you upload `model.tar`, then run the notebook top-to-bottom.

## 0. Inspect the attached GPU
Verify that the vast.ai instance exposes the accelerators you rented before configuring the training run.

In [ ]:
!nvidia-smi


## 1. Configure repository checkout and training plan
Fill in the variables below:

- `REPO_URL` should point to your DurakZero fork (add a token if the repo is private).
- `BRANCH_NAME` is the branch with the latest Durak-focused code (the default `work` branch tracks this notebook).
- `MODEL_TAR_SOURCE` is the path where you upload the most recent `model.tar` checkpoint (if empty, training starts from scratch).
- GPU/workload parameters default to a 2× RTX 5090 instance (Type #27547157) with 32 CPU cores; adjust them if you rent a different configuration.

The derived values, especially `CHECKPOINT_ROOT` and `MODEL_DEST`, control where checkpoints are stored on disk.

In [ ]:
from pathlib import Path
import torch

# === Repository inputs ===
REPO_URL = "https://github.com/YOUR_USERNAME/DurakZero.git"  # TODO: update
BRANCH_NAME = "work"
WORKDIR = Path("/workspace/durakzero")
MODEL_TAR_SOURCE = Path("/workspace/input/model.tar")  # TODO: update to your upload location
XP_ID = "vastai_run"

# === Resource plan (defaults assume 2× RTX 5090 with 32 CPU cores) ===
NUM_GPUS_TO_USE = 2
ACTORS_PER_DEVICE = 10
BATCH_SIZE = 64
UNROLL_LENGTH = 160
NUM_BUFFERS = 120
NUM_THREADS = 4
SAVE_INTERVAL_MIN = 15
LEARNING_RATE = 7e-5
TRAINING_DEVICE_INDEX = 0  # learner stays on the first listed GPU

# Derived paths and sanity checks
if "YOUR_USERNAME" in REPO_URL:
    raise ValueError("Set REPO_URL to point at your DurakZero fork (replace YOUR_USERNAME)")

available_gpus = torch.cuda.device_count()
if NUM_GPUS_TO_USE > available_gpus:
    raise ValueError(f"Requested {NUM_GPUS_TO_USE} GPUs but only {available_gpus} detected")

gpu_devices = ",".join(str(i) for i in range(NUM_GPUS_TO_USE))
CHECKPOINT_ROOT = WORKDIR / "durakzero_checkpoints"
MODEL_DEST = CHECKPOINT_ROOT / XP_ID / "model.tar"

TRAINING_FLAG_ITEMS = [
    ("--xpid", XP_ID),
    ("--save_interval", SAVE_INTERVAL_MIN),
    ("--gpu_devices", gpu_devices),
    ("--num_actor_devices", NUM_GPUS_TO_USE),
    ("--num_actors", ACTORS_PER_DEVICE),
    ("--training_device", TRAINING_DEVICE_INDEX),
    ("--batch_size", BATCH_SIZE),
    ("--unroll_length", UNROLL_LENGTH),
    ("--num_buffers", NUM_BUFFERS),
    ("--num_threads", NUM_THREADS),
    ("--learning_rate", LEARNING_RATE),
    ("--savedir", str(CHECKPOINT_ROOT)),
]

print(f"Repository URL      : {REPO_URL}")
print(f"Branch              : {BRANCH_NAME}")
print(f"Work directory      : {WORKDIR}")
print(f"Using GPU indices   : {gpu_devices}")
print(f"Actors per device   : {ACTORS_PER_DEVICE}")
print(f"Total actors        : {NUM_GPUS_TO_USE * ACTORS_PER_DEVICE}")
print(f"Batch size          : {BATCH_SIZE}")
print(f"Unroll length       : {UNROLL_LENGTH}")
print(f"Buffers per device  : {NUM_BUFFERS}")
print(f"Learner threads/dev : {NUM_THREADS}")
print(f"Checkpoint root     : {CHECKPOINT_ROOT}")
print(f"Expected checkpoint : {MODEL_DEST}")
print("Uploaded model.tar  :", MODEL_TAR_SOURCE.exists())


## 2. Clone your fork and place checkpoints
The cell below wipes any previous checkout, clones your branch, and (optionally) seeds the checkpoint directory with an uploaded `model.tar` so training resumes from the latest weights.

In [ ]:
import os
import shutil
import subprocess

if WORKDIR.exists():
    print(f"Removing existing directory: {WORKDIR}")
    shutil.rmtree(WORKDIR)

clone_cmd = ["git", "clone", "--branch", BRANCH_NAME, REPO_URL, str(WORKDIR)]
print("Cloning repo:\n ", " \n".join(clone_cmd))
subprocess.run(clone_cmd, check=True)

os.chdir(WORKDIR)
print("Changed working directory to", WORKDIR)


In [ ]:
import shutil

MODEL_DEST.parent.mkdir(parents=True, exist_ok=True)
if MODEL_TAR_SOURCE.exists():
    shutil.copy2(MODEL_TAR_SOURCE, MODEL_DEST)
    print(f"Copied {MODEL_TAR_SOURCE} -> {MODEL_DEST}")
else:
    print(f"No model.tar found at {MODEL_TAR_SOURCE}; starting from scratch")


## 3. Install Python dependencies with `uv`
The `vastai/pytorch` image ships with PyTorch, but DurakZero also relies on `gitpython` for logging metadata and standard scientific packages. These commands install the extras and register the repo in editable mode.

In [ ]:
!pip install --quiet --upgrade pip uv
!uv pip install --system gitpython numpy
!uv pip install --system -e .


## 4. Assemble and inspect the training command
This cell builds the `train.py` invocation that matches your resource plan. Review the command string and confirm that `--load_model` appears when a checkpoint is available.

In [ ]:
import shlex

LOAD_EXISTING = MODEL_DEST.exists()
cmd = ["python", "train.py"]
for flag, value in TRAINING_FLAG_ITEMS:
    cmd.extend([flag, str(value)])
if LOAD_EXISTING:
    cmd.append("--load_model")

print("Training command:\n ", " \n".join(shlex.quote(part) for part in cmd))
print(f"Loading existing checkpoint: {LOAD_EXISTING}")


## 5. Launch training
Run the command assembled above. The process will stream logs (including rolling win-rates) to the notebook; stop the cell manually when you reach your 4–5 hour budget. Checkpoints land in `CHECKPOINT_ROOT / XP_ID`.

> Tip: keep an eye on GPU utilisation in another terminal (`watch -n 5 nvidia-smi`) or open a new notebook cell to monitor system metrics while this cell runs.

In [ ]:
import subprocess

process = subprocess.Popen(cmd, cwd=str(WORKDIR))
process.wait()
